# Stress Académique — Exploration et Résultats

**Rôle de ce notebook :** visualisation, compréhension des enjeux et interprétation des résultats.  
L'analyse complète (entraînement, tests économétriques) est exécutée dans les scripts Python :
- `scripts/run_analysis.py`
- `scripts/run_econometrics.py`

---

**Contexte :** 140 étudiants interrogés sur leurs sources de stress (juillet–août 2025).  
L'objectif est d'identifier les facteurs prédicteurs d'un **stress académique élevé** (indice ≥ 4/5)  
et de construire un outil de dépistage précoce.

> Exécutez d'abord `python scripts/run_analysis.py` pour générer les sorties dans `outputs/`.

In [ ]:
import sys
from pathlib import Path

# Ajouter la racine du projet au path
_ROOT = Path().resolve().parent
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns
from IPython.display import display

from src.config import get_config, resolve_path
from src.data_loader import load_and_clean_data, get_data_summary
from src.preprocessing import extract_features_target

plt.style.use('seaborn-v0_8-whitegrid')
PALETTE = sns.color_palette('muted')
cfg = get_config()

print('Configuration chargée.')

## 1. Chargement et aperçu des données

In [ ]:
data_path = resolve_path(cfg['data']['path'])
df, report = load_and_clean_data(data_path)

print(report.summary())
print(f'\nDimensions finales : {df.shape[0]} lignes × {df.shape[1]} colonnes')

In [ ]:
# Aperçu des premières lignes
display(df.head(5))

In [ ]:
# Statistiques descriptives des variables numériques
ordinal_cols = cfg['columns']['ordinal'] + [cfg['data']['target_column']]
display(df[ordinal_cols].describe().round(2))

## 2. Distribution des variables

Les étudiants rapportent en moyenne un stress **au-dessus du point médian** (3.72/5).  
La compétition académique (3.48) et la pression familiale (3.17) sont également élevées.

In [ ]:
target_col = cfg['data']['target_column']
cutoff = cfg['target']['high_stress_cutoff']

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
cols_to_plot = ordinal_cols

for ax, col in zip(axes, cols_to_plot):
    counts = df[col].value_counts().sort_index()
    colors = ['#d62728' if v >= cutoff else '#1f77b4' for v in counts.index]
    ax.bar(counts.index.astype(str), counts.values, color=colors, alpha=0.85, edgecolor='white')
    ax.set_title(col[:35], fontsize=9)
    ax.set_xlabel('Score (1–5)')
    ax.set_ylabel('Effectif')
    if col == target_col:
        ax.axvline(str(cutoff - 0.5), color='red', ls='--', lw=1.5, label=f'Seuil ≥{cutoff}')
        ax.legend(fontsize=8)

fig.suptitle('Distribution des variables ordinales\n(rouge = score ≥ 4, zone haut-stress)', y=1.02)
fig.tight_layout()
plt.show()

In [ ]:
nominal_cols = cfg['columns']['nominal']

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()

for ax, col in zip(axes, nominal_cols):
    counts = df[col].value_counts()
    ax.barh(counts.index, counts.values, color=PALETTE[:len(counts)], alpha=0.85)
    ax.set_title(col[:55], fontsize=9)
    ax.set_xlabel('Effectif')
    for i, v in enumerate(counts.values):
        ax.text(v + 0.3, i, str(v), va='center', fontsize=9)

fig.suptitle('Distribution des variables catégorielles nominales')
fig.tight_layout()
plt.show()

## 3. Analyse bivariée — Stress selon les groupes

Les box plots montrent l'indice de stress médian selon chaque modalité des variables catégorielles.  
Les différences les plus marquées s'observent pour l'**environnement d'étude** et les **habitudes néfastes**.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
axes = axes.flatten()

for ax, col in zip(axes, nominal_cols):
    order = df.groupby(col)[target_col].median().sort_values(ascending=False).index
    sns.boxplot(
        data=df, x=col, y=target_col,
        order=order, palette='muted', ax=ax,
        width=0.5,
    )
    ax.set_title(f'Stress selon : {col[:40]}', fontsize=9)
    ax.set_xlabel('')
    ax.set_ylabel('Indice de stress (1–5)')
    ax.tick_params(axis='x', rotation=15, labelsize=8)
    ax.axhline(4, color='red', ls='--', lw=1, alpha=0.6, label='Seuil HS')

fig.suptitle('Distribution du stress académique par groupe')
fig.tight_layout()
plt.show()

## 4. Corrélations entre variables ordinales

La **compétition académique** (ρ ≈ 0.45) et la **pression des pairs** (ρ ≈ 0.47) présentent  
les corrélations de Spearman les plus fortes avec l'indice de stress.

In [ ]:
from scipy.stats import spearmanr

corr_cols = cfg['columns']['ordinal'] + [target_col]
n = len(corr_cols)
rho_matrix = pd.DataFrame(np.ones((n, n)), index=corr_cols, columns=corr_cols)

for i, c1 in enumerate(corr_cols):
    for j, c2 in enumerate(corr_cols):
        if i != j:
            rho, _ = spearmanr(df[c1], df[c2])
            rho_matrix.loc[c1, c2] = round(rho, 2)

# Raccourcir les noms pour la lisibilité
aliases = {
    'Peer pressure': 'Pression pairs',
    'Academic pressure from your home': 'Pression familiale',
    'What would you rate the academic  competition in your student life': 'Compétition',
    target_col: 'Stress (cible)',
}
rho_short = rho_matrix.copy()
rho_short.index = [aliases.get(c, c) for c in rho_matrix.index]
rho_short.columns = [aliases.get(c, c) for c in rho_matrix.columns]

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    rho_short.astype(float), annot=True, fmt='.2f',
    cmap='RdBu_r', center=0, vmin=-1, vmax=1,
    linewidths=0.5, ax=ax,
)
ax.set_title('Matrice de corrélations de Spearman')
fig.tight_layout()
plt.show()

## 5. Variable cible — Prévalence du haut stress

Avec un seuil de 4/5, **environ 70 %** des répondants sont classés à haut stress.

In [ ]:
X, y = extract_features_target(df)
n_pos = int(y.sum())
n_neg = int((y == 0).sum())
prev = float(y.mean())

fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(['Faible stress\n(score < 4)', f'Haut stress\n(score ≥ {cutoff})'],
       [n_neg, n_pos],
       color=['#1f77b4', '#d62728'], alpha=0.85, width=0.5)
ax.bar_label(ax.containers[0], fmt='%d', padding=3)
ax.set_ylabel('Effectif')
ax.set_title(f'Prévalence du haut stress\n({prev:.1%} de l\'échantillon)')
fig.tight_layout()
plt.show()

print(f'Haut stress (y=1) : {n_pos} étudiants ({prev:.1%})')
print(f'Faible stress (y=0): {n_neg} étudiants ({1-prev:.1%})')

## 6. Résultats du modèle

Les figures suivantes sont générées par `scripts/run_analysis.py`.  
Assurez-vous d'avoir exécuté ce script avant de continuer.

In [ ]:
fig_dir = resolve_path(cfg['outputs']['figures_dir'])
results_dir = resolve_path(cfg['outputs']['results_dir'])

def show_figure(fname, title=''):
    path = fig_dir / fname
    if not path.exists():
        print(f'Figure non trouvée : {path}\nExécutez scripts/run_analysis.py d\'abord.')
        return
    img = mpimg.imread(str(path))
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.imshow(img)
    ax.axis('off')
    if title:
        ax.set_title(title, fontsize=11)
    plt.tight_layout()
    plt.show()

In [ ]:
show_figure('roc_curve.png', 'Courbe ROC — Régression logistique')

In [ ]:
show_figure('pr_curve.png', 'Courbe Précision-Rappel')

In [ ]:
show_figure('confusion_matrix.png', 'Matrice de confusion (seuil opérationnel)')

In [ ]:
show_figure('odds_ratios.png', 'Odds Ratios bootstrap (IC 95 %)')

In [ ]:
show_figure('threshold_comparison.png', 'Comparaison des seuils de décision')

## 7. Résumé des métriques

Chargement des résultats depuis `outputs/results/full_results.json`.

In [ ]:
results_path = results_dir / 'full_results.json'

if results_path.exists():
    with open(results_path) as f:
        results = json.load(f)

    m = results['metrics_test']
    cv = results['cv_results']

    print('=== Performance sur le jeu de test (seuil opérationnel) ===')
    print(f'  AUC-ROC      : {m["roc_auc"]:.3f}')
    print(f'  PR-AUC       : {m["pr_auc"]:.3f}')
    print(f'  Brier score  : {m["brier_score"]:.4f}')
    print(f'  Accuracy     : {m["accuracy"]:.3f}')
    print(f'  Sensibilité  : {m["sensitivity"]:.3f}')
    print(f'  Spécificité  : {m["specificity"]:.3f}')
    print(f'  F1           : {m["f1"]:.3f}')
    print(f'  TP={m["tp"]}  TN={m["tn"]}  FP={m["fp"]}  FN={m["fn"]}')

    print('\n=== Validation croisée (5-fold) ===')
    for metric, stats in cv.items():
        print(f'  {metric:<20}: {stats["mean"]:.3f} ± {stats["std"]:.3f}')
else:
    print('Résultats non trouvés. Exécutez scripts/run_analysis.py d\'abord.')

## 8. Résultats économétriques clés

Chargement depuis `outputs/results/econometrics_report.json`.

In [ ]:
eco_path = results_dir / 'econometrics_report.json'

if eco_path.exists():
    with open(eco_path) as f:
        eco = json.load(f)

    pr2 = eco['pseudo_r2']
    ic  = eco['information_criteria']
    lr  = eco['likelihood_ratio_test']
    hl  = eco['hosmer_lemeshow']

    print('=== Pseudo-R² ===')
    print(f'  McFadden  : {pr2["mcfadden_r2"]:.4f}  (> 0.20 = bon ajustement)')
    print(f'  Cox-Snell : {pr2["cox_snell_r2"]:.4f}')
    print(f'  Nagelkerke: {pr2["nagelkerke_r2"]:.4f}')

    print('\n=== Critères d\'information ===')
    print(f'  AIC : {ic["aic"]:.2f}')
    print(f'  BIC : {ic["bic"]:.2f}')

    print('\n=== Likelihood Ratio Test ===')
    print(f'  stat={lr["statistic"]:.3f}  p={lr["p_value"]:.4f}  df={lr["df"]}')
    print(f'  → {lr["conclusion"]}')

    print('\n=== Hosmer-Lemeshow ===')
    print(f'  stat={hl["statistic"]:.3f}  p={hl["p_value"]:.4f}  df={hl["df"]}')
    print(f'  → {hl["conclusion"]}')

    print('\n=== Top 5 effets marginaux ===')
    me_df = pd.DataFrame(eco['marginal_effects']).head(5)
    display(me_df[['feature', 'coef', 'AME']])
else:
    print('Rapport économétrique non trouvé. Exécutez scripts/run_econometrics.py d\'abord.')

In [ ]:
show_figure('vif_barplot.png', 'VIF — Détection de multicolinéarité')

In [ ]:
show_figure('marginal_effects.png', 'Effets marginaux moyens (AME)')

## 9. Synthèse et recommandations

### Principaux enseignements

| Facteur | Odds Ratio | Interprétation |
|---|---|---|
| Compétition académique élevée (4–5) | ~3.0 | **Facteur de risque #1** — risque triplé |
| Pression des pairs élevée (4–5) | ~2.2 | Risque doublé |
| Pression familiale élevée (4–5) | ~1.8 | Risque +80 % |

### Performance du modèle

- **AUC-ROC = 0.747** — discrimination bonne (au-dessus du hasard de 25 pp)
- **Sensibilité = 0.806** au seuil opérationnel — 80 % des cas haut-stress détectés
- **Brier score = 0.203** — probabilités raisonnablement calibrées

### Seuil opérationnel recommandé : **t = 0.465**

- Sensibilité : 0.806 / Spécificité : 0.611
- Compromis approprié pour un usage de dépistage : manquer un étudiant à risque
  (faux-négatif) est considéré **3× plus coûteux** qu'une fausse alarme.

### Limites à retenir

- Échantillon de convenance (n=139) — résultats non généralisables
- Données auto-rapportées — biais de désirabilité sociale possible
- Modèle linéaire — interactions entre facteurs non modélisées